# PersonaPlex — RunPod RTX 5090 Deployment Notebook

This notebook deploys **[PersonaPlex](https://github.com/NVIDIA/personaplex)** — NVIDIA's real-time, full-duplex
speech-to-speech model with persona/voice control (built on [Moshi](https://arxiv.org/abs/2410.00037)) — on a
**fresh RunPod pod with an RTX 5090 GPU**.

It is derived directly from the repository's own `README.md`, `client/README.md`, `moshi/pyproject.toml`,
`moshi/requirements.txt`, and the server/offline entrypoints (`moshi/moshi/server.py`, `moshi/moshi/offline.py`,
`moshi/moshi/utils/connection.py`, `moshi/moshi/models/loaders.py`). No steps are invented — every command below
maps to something the repo actually does or documents.

## What this notebook does

1. Verifies the RunPod environment, GPU and CUDA.
2. Sets up persistent storage on the RunPod volume.
3. Installs system + Python dependencies (including the **Blackwell/RTX 5090‑specific PyTorch build**).
4. Clones the repository (skipped automatically if you already uploaded it).
5. Configures Hugging Face authentication and downloads the model weights, tokenizer, voices and web UI assets.
6. Starts the PersonaPlex backend server (which also serves the prebuilt web UI — no separate frontend build is
   required for normal use).
7. Verifies the deployment with an automated offline inference smoke test and an HTTP check.
8. Documents GPU optimization notes and troubleshooting steps.

## What PersonaPlex does *not* have (so these checklist items are intentionally empty)

- **No database** of any kind — there is nothing to initialize.
- **No separate "API server"** — the single aiohttp server in `moshi/moshi/server.py` exposes both the WebSocket
  endpoint (`/api/chat`) and the static web UI on one port.
- **No Gradio/Streamlit app** — the only Gradio dependency is `gradio.networking.setup_tunnel`, an *optional*
  public-URL tunnel (`--gradio-tunnel`), not a Gradio UI. RunPod's own HTTP port proxy makes this unnecessary here.
- **No required "configuration file" to generate** — runtime behavior is controlled entirely by CLI flags and the
  HF-hosted `config.json` that ships with the model weights.

## Things this notebook *cannot* automate (you must do these yourself)

1. **Accept the NVIDIA Open Model License** for the gated model repo
   [`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1) — log into Hugging Face in a
   browser and click "Agree and access repository". No script can click this for you.
2. **Create a Hugging Face access token** for that same account and have it ready to paste into the auth cell
   below.
3. **Expose the server port (default `8998`) as an HTTP Service** from the RunPod pod's *Connect* page (RunPod
   console action) so the proxy URL works from outside the pod.
4. **Grant microphone permission** in your browser when you open the live web UI — this is a per-user browser
   security prompt.

Run the cells **top to bottom**. The only cell you must intentionally run out of the normal flow is the final
**"Stop the server"** cleanup cell — leave it for whenever you actually want to shut the server down.

## 1. Environment sanity checks

Confirms this is a Linux pod with a GPU attached and a supported Python version (`moshi-personaplex` requires
Python ≥ 3.10, per `moshi/pyproject.toml`).

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

assert sys.version_info >= (3, 10), (
    f"PersonaPlex (moshi/pyproject.toml) requires Python >= 3.10, found {sys.version_info}."
)
print("Python version OK.")

In [ ]:
# Confirm the NVIDIA driver sees a GPU at the OS level before we install anything.
!nvidia-smi

## 2. Persistent storage setup (RunPod volume)

RunPod mounts your persistent Network Volume at **`/workspace`**. Anything written there survives pod
stop/start (model weights are multiple GB, so re-downloading them on every restart would be wasteful).
If `/workspace` isn't present (e.g. you're running this notebook somewhere else), we fall back to the home
directory so the notebook still works end-to-end.

We also point Hugging Face's cache (`HF_HOME`) at the persistent volume so `huggingface_hub` downloads
(triggered both by this notebook and internally by `moshi/moshi/server.py` / `moshi/moshi/offline.py`) are
cached once and reused across restarts.

In [ ]:
import os

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-RAG-code-streaming-v2.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"   # loaders.DEFAULT_REPO in moshi/moshi/models/loaders.py

SERVER_HOST = "0.0.0.0"   # must be 0.0.0.0 (not "localhost") so RunPod's proxy can reach the server
SERVER_PORT = 8998        # default port used by moshi.server

# RunPod's HTTP port proxy terminates TLS at its edge and forwards plain HTTP to the container, so by
# default we do NOT enable the app's own self-signed TLS (--ssl). Flip this to True only if you plan to
# expose SERVER_PORT directly as a raw TCP port instead of through RunPod's HTTP proxy.
USE_APP_TLS = False

# An RTX 5090 has 32GB of VRAM, comfortably enough for this 7B model in bf16 -- CPU offload should not be
# needed. Flip to True only if you hit CUDA OOM (requires the `accelerate` package, installed below anyway).
USE_CPU_OFFLOAD = False

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
# Keep PATH aware of ~/.local/bin, where moshi/moshi/utils/connection.py installs `mkcert` if --ssl is used.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

print("WORKSPACE   :", WORKSPACE)
print("REPO_DIR    :", REPO_DIR)
print("HF_HOME     :", os.environ["HF_HOME"])
print("USE_APP_TLS :", USE_APP_TLS)

## 3. System package installation

Per the repo's `README.md` prerequisites: install the **Opus codec development library** before installing the
Python package (it's required by `sphn`, which PersonaPlex uses for Opus audio streaming over the WebSocket).
We also make sure `git` is present for cloning.

In [ ]:
import os

SUDO = "" if os.geteuid() == 0 else "sudo "

!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")

## 4. Repository cloning

If `REPO_DIR` doesn't already contain the project (e.g. you uploaded your own copy of this repo to the volume
beforehand), it is cloned fresh from GitHub. If it's already there, cloning is skipped automatically — this
cell is safe to re-run on every pod restart.

In [ ]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"

if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

assert repo_marker.exists(), f"Expected {repo_marker} to exist after cloning/upload."


## 5. Python dependency installation

The repo's documented install command is:

```bash
pip install moshi/.
```

which installs from `moshi/pyproject.toml` (`numpy`, `safetensors`, `huggingface-hub`, `einops`,
`sentencepiece`, `sounddevice`, `sphn`, `torch>=2.2,<2.5`, `aiohttp`).

### RTX 5090 / Blackwell note

The pinned `torch<2.5` build does **not** ship CUDA kernels for Blackwell (RTX 50‑series, `sm_120`) GPUs. The
repo's `README.md` explicitly documents the fix for this
([NVIDIA/personaplex#2](https://github.com/NVIDIA/personaplex/issues/2)): reinstall PyTorch from the `cu130`
wheel index *after* the base install. This intentionally overrides the `<2.5` pin — that's expected and is the
upstream-recommended fix, not a mistake.

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/." 

In [ ]:
# Blackwell (RTX 5090) requires CUDA-13.0-built PyTorch wheels. This intentionally supersedes the
# torch<2.5 pin from moshi/pyproject.toml -- see README.md "Extra step for Blackwell based GPUs".
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# `accelerate` enables --cpu-offload (README's "CPU Offload" section); `gradio` enables the optional
# --gradio-tunnel fallback for public exposure if you ever need it instead of RunPod's HTTP proxy.
%pip install -q accelerate gradio

## 6. CUDA / GPU verification

Confirms PyTorch can see the RTX 5090 and that the installed build actually has working CUDA kernels for it
(a bf16 matmul smoke test) — this is the check that would have failed before the `cu130` reinstall above if it
had been skipped.

In [ ]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
print("CUDA available     :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected by PyTorch. Confirm this RunPod pod has an RTX 5090 GPU attached "
        "and that the driver is healthy (see the `nvidia-smi` output above)."
    )

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU                :", device_name)
print("Compute capability :", capability)

if "5090" not in device_name and "RTX 50" not in device_name:
    print(f"WARNING: expected an RTX 5090, found '{device_name}'. Continuing anyway.")

# Smoke test: this is exactly the kind of op that fails with
# "no kernel image is available for execution on the device" on Blackwell if torch wasn't
# reinstalled from the cu130 index.
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
y = x @ x
torch.cuda.synchronize()
print("bf16 CUDA matmul smoke test OK, result shape:", tuple(y.shape))

## 7. Hugging Face authentication

**Manual step required before running this cell:** log into Hugging Face in a browser, open
[`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), and click **"Agree and access
repository"** to accept the NVIDIA Open Model License. Then create an access token (read access is enough) at
<https://huggingface.co/settings/tokens>.

The repo's `README.md` documents this as `export HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>`; the cell below does the
same thing from inside the notebook, via a hidden prompt so the token isn't echoed into cell output.

In [ ]:
from getpass import getpass

from huggingface_hub import login

# hf_token = os.environ.get("HF_TOKEN")
# sa token...not mine
hf_token = 'tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt' 
if not hf_token:
    hf_token = getpass("Enter your Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

## 8. Model downloading

`moshi/moshi/server.py` and `moshi/moshi/offline.py` both lazily call `hf_hub_download` for each asset the
first time they need it. We pre-fetch the same files here so that (a) any license/token problem surfaces now
with a clear error instead of mid-startup, and (b) everything is already warm in the `HF_HOME` cache before we
launch the server.

Assets (from `moshi/moshi/models/loaders.py` and `moshi/moshi/server.py`):
- `config.json` — model config
- `tokenizer_spm_32k_3.model` — SentencePiece text tokenizer
- `tokenizer-e351c8d8-checkpoint125.safetensors` — Mimi codec weights
- `model.safetensors` — Moshi/PersonaPlex LM weights
- `voices.tgz` — packaged voice-prompt embeddings (NATF0‑3, NATM0‑3, VARF0‑4, VARM0‑4)
- `dist.tgz` — the **prebuilt web UI** (this is why no separate frontend build step is needed)

In [ ]:
import tarfile

from huggingface_hub import hf_hub_download

ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
    "dist.tgz",
]

downloaded = {}
try:
    for fname in ASSET_FILES:
        # No explicit cache_dir: this honors HF_HOME (set in step 2) so the notebook and the server
        # subprocess we launch later (which inherits the same env) share one cache.
        path = hf_hub_download(HF_REPO_ID, fname)
        downloaded[fname] = path
        print(f"OK  {fname} -> {path}")
except Exception as e:
    raise RuntimeError(
        "Failed to download model assets from "
        f"https://huggingface.co/{HF_REPO_ID}. This almost always means either:\n"
        "  1) you have not clicked 'Agree and access repository' on that model page yet, or\n"
        "  2) the HF_TOKEN you supplied doesn't belong to the account that accepted the license, or\n"
        "  3) the token is invalid/expired.\n"
        f"Original error: {e}"
    )

In [ ]:
import pathlib

# Pre-extract the tarballs once, exactly like _get_voice_prompt_dir / _get_static_path do in
# moshi/moshi/server.py, so the first real request doesn't pay the extraction cost.
for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    if not out_dir.exists():
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir} ({'already extracted' if out_dir.exists() else 'extracted now'})")

## 9. TLS certificate directory (only used if `USE_APP_TLS = True`)

The README's default local-machine workflow is:

```bash
SSL_DIR=$(mktemp -d); python -m moshi.server --ssl "$SSL_DIR"
```

`--ssl` makes `moshi/moshi/utils/connection.py` auto-install `mkcert` and generate a **self-signed**
certificate in that directory. That's appropriate for direct LAN/local access, but on RunPod we're putting the
server behind RunPod's own HTTPS proxy (Section 11), which already terminates TLS at the edge — adding a second,
self-signed TLS layer underneath it would just produce certificate warnings for no benefit. We still create the
directory here so you can flip `USE_APP_TLS = True` above if you'd rather expose `SERVER_PORT` as a raw TCP port
instead of through the HTTP proxy.

In [ ]:
import tempfile

SSL_DIR = tempfile.mkdtemp(prefix="personaplex_ssl_")
print("SSL_DIR:", SSL_DIR, "(only used if USE_APP_TLS = True)")

## 10. Backend (+ web UI) startup

This is the repo's **recommended and only documented launch method** — `python -m moshi.server`. It is a
single aiohttp process that serves:
- the WebSocket endpoint `/api/chat` (the real-time speech protocol), and
- the prebuilt web UI from `dist.tgz` at `/` (downloaded in Section 8) — there is **no separate frontend
  process to start** for normal use. (The `client/` folder in the repo is only needed if you want to modify the
  React/Vite frontend source and rebuild it yourself — see `client/README.md`.)

A notebook cell that calls `web.run_app(...)` directly would block the kernel forever, so we launch it as a
background subprocess and log to a file, then poll that log in the next cell.

In [ ]:
import subprocess
import sys

env = os.environ.copy()
LOG_PATH = os.path.join(WORKSPACE, "personaplex_server.log")

cmd = [sys.executable, "-m", "moshi.server", "--host", SERVER_HOST, "--port", str(SERVER_PORT)]
if USE_APP_TLS:
    cmd += ["--ssl", SSL_DIR]
if USE_CPU_OFFLOAD:
    cmd += ["--cpu-offload"]

print("Launching:", " ".join(cmd))

log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Server launched with PID {server_proc.pid}. Logs: {LOG_PATH}")

In [ ]:
import time

def tail(path, n=60):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI directly at"
TIMEOUT_S = 900  # first run downloads + loads a multi-GB model; be generous
POLL_S = 5

start = time.time()
ready = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print(tail(LOG_PATH))
        raise RuntimeError(
            f"Server process exited early with return code {server_proc.returncode}. See log above."
        )
    if READY_MARKER in open(LOG_PATH).read():
        ready = True
        break
    time.sleep(POLL_S)

print(tail(LOG_PATH))

if not ready:
    raise TimeoutError(
        f"Server did not report readiness within {TIMEOUT_S}s. Check the log tail above -- "
        "the most common causes are a slow first-time model download or a CUDA/driver mismatch."
    )

print("\nServer is up and warmed up.")

## 11. Expose the port & get the access URL

RunPod will not route external traffic to a port unless you explicitly expose it. In the pod's **Connect**
page, add an **HTTP Service** for `SERVER_PORT` (default `8998`) if you haven't already. RunPod then serves it
at `https://<POD_ID>-<PORT>.proxy.runpod.net`, with RunPod terminating TLS — which is exactly why
`USE_APP_TLS = False` is the right default here (see Section 9).

In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID")

if pod_id:
    public_url = f"https://{pod_id}-{SERVER_PORT}.proxy.runpod.net"
    print("RunPod public URL (requires SERVER_PORT to be exposed as an HTTP Service on the pod's Connect page):")
    print(" ", public_url)
else:
    print(
        "RUNPOD_POD_ID was not found in the environment. If you are on RunPod, expose "
        f"port {SERVER_PORT} as an HTTP Service from the pod's Connect page and use the proxy URL shown there."
    )

scheme = "https" if USE_APP_TLS else "http"
print(f"Local URL inside the pod: {scheme}://localhost:{SERVER_PORT}")

## 12. Verification — offline inference smoke test

This runs the repo's documented `moshi.offline` "Assistant example" exactly as given in `README.md` — it
streams `assets/test/input_assistant.wav` through the model with voice prompt `NATF2.pt` and writes an output
WAV + JSON transcript. This doesn't need a browser, microphone, or open port, so it's a good way to confirm the
backend, weights and GPU are all working correctly before testing the live UI.

In [ ]:
import subprocess
import sys

offline_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", "NATF2.pt",
    "--input-wav", "assets/test/input_assistant.wav",
    "--seed", "42424242",
    "--output-wav", "output_assistant.wav",
    "--output-text", "output_assistant.json",
]

result = subprocess.run(
    offline_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True,
)

print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("Offline smoke test failed -- see output above.")

print("\nOffline smoke test succeeded.")

In [ ]:
import json

from IPython.display import Audio, display

output_wav_path = os.path.join(REPO_DIR, "output_assistant.wav")
output_json_path = os.path.join(REPO_DIR, "output_assistant.json")

display(Audio(output_wav_path))

with open(output_json_path) as f:
    tokens = json.load(f)
print("Generated text tokens (joined):")
print("".join(tokens))

## 13. Verification — HTTP check on the live server

Confirms the running server process answers on `/` (the web UI's `index.html`, served from `dist.tgz`).

In [ ]:
import ssl
import urllib.request

scheme = "https" if USE_APP_TLS else "http"
ctx = ssl._create_unverified_context() if USE_APP_TLS else None

with urllib.request.urlopen(f"{scheme}://localhost:{SERVER_PORT}/", context=ctx, timeout=15) as resp:
    print("HTTP status :", resp.status)
    print("Content-Type:", resp.headers.get("Content-Type"))
    assert resp.status == 200, f"Expected 200, got {resp.status}"

print("Web UI is being served correctly.")

## 14. Using the live web UI

Open the URL printed in Section 11 in a browser, allow microphone access when prompted (this is the one
browser permission step that can't be automated), and start talking.

### Voices (from `README.md`)

```
Natural(female): NATF0, NATF1, NATF2, NATF3
Natural(male):   NATM0, NATM1, NATM2, NATM3
Variety(female): VARF0, VARF1, VARF2, VARF3, VARF4
Variety(male):   VARM0, VARM1, VARM2, VARM3, VARM4
```

### Example role prompts (from `README.md`)

- Assistant role: `You are a wise and friendly teacher. Answer questions or provide advice in a clear and
  engaging way.`
- Casual conversation: `You enjoy having a good conversation.`
- Customer service example: `You work for CitySan Services which is a waste management company and your name
  is Ayelen Lucero. Information: Verify customer name Omar Torres. Current schedule: every other week.
  Upcoming pickup: April 12th. Compost bin service available for $8/month add-on.`

See the repo's `README.md` "Prompting Guide" section for more examples and guidance.

## 15. GPU optimization notes for RTX 5090

- **bf16 by default**: `moshi/moshi/models/loaders.py:get_moshi_lm` loads the LM in `torch.bfloat16`. Blackwell
  has fast native bf16 Tensor Core throughput, so no dtype changes are needed.
- **`--cpu-offload` is unnecessary here**: it exists in `server.py`/`offline.py` for GPUs with insufficient
  VRAM (via `accelerate`'s `infer_auto_device_map`). An RTX 5090's 32GB comfortably fits this 7B model plus the
  Mimi codec; only enable `USE_CPU_OFFLOAD` if you're also running other large workloads on the same GPU.
- **One process = one GPU's worth of model**: `ServerState.__init__` loads two Mimi instances and the LM once
  per process and keeps them resident (`streaming_forever`). Don't launch a second `moshi.server` process
  against the same GPU unless you've confirmed there's VRAM headroom — check with `nvidia-smi`.
- **Warmup cost is already handled**: `state.warmup()` runs 4 dummy frames through the full encode → LM →
  decode path right after model load, ahead of any real connections — that's the per-process latency you saw
  the log wait for in Section 10, not something to optimize further.
- **CUDA build must match the GPU**: Blackwell (`sm_120`) needs the `cu130` PyTorch wheels installed in
  Section 5. If you ever see `no kernel image is available for execution on the device`, that cell needs to be
  re-run (something likely reinstalled a different torch build afterward).

## 16. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `401`/`403` downloading model assets | License not accepted, or `HF_TOKEN` doesn't belong to the account that accepted it | Re-check Section 7: accept the license at the model page with the **same** account whose token you pasted |
| `no kernel image is available for execution on the device` | Torch build doesn't have Blackwell (`sm_120`) kernels | Re-run the `cu130` reinstall cell in Section 5, then re-run Section 6's smoke test |
| `ImportError` / build errors mentioning `opus` while installing `sphn` | `libopus-dev` missing | Re-run Section 3, then re-run Section 5 |
| Server process exits immediately, log shows a CUDA OOM | Not enough free VRAM (e.g. another process holds the GPU) | Check `nvidia-smi`; consider `USE_CPU_OFFLOAD = True` (Section 2) and re-run Sections 10‑11 |
| Browser blocks microphone / `getUserMedia` fails | Page wasn't loaded over a secure context | Use the RunPod **proxy** URL from Section 11 (HTTPS at the edge), not a plain `http://<pod-ip>:8998` URL |
| Can't reach the URL from outside the pod at all | Port not exposed | In the RunPod console, add `SERVER_PORT` as an HTTP Service on the pod's Connect page |
| First launch seems to hang for several minutes | Normal — first run downloads multi-GB weights and extracts `voices.tgz`/`dist.tgz` | Watch the log tail printed by Section 10; increase `TIMEOUT_S` if your network is slow |
| `mkcert` warnings in the log | Only relevant when `USE_APP_TLS = True`; `moshi/moshi/utils/connection.py` falls back to plain HTTP automatically if `mkcert` can't be installed | Safe to ignore in default (proxy) mode |

### Recap: what genuinely cannot be automated by this notebook
1. Clicking "Agree and access repository" on the gated HF model page.
2. Issuing the HF access token for that account.
3. Exposing `SERVER_PORT` as an HTTP Service in the RunPod console.
4. The browser's microphone-permission prompt.

## 18. RAG research configuration (optional — disabled by default)

This section adds the configuration surface for the **RAG / runtime knowledge injection research
platform** described in `docs/ARCHITECTURE_REPORT.md` and `docs/STREAMING_AND_INJECTION_DESIGN.md`.

**`ENABLE_RAG = False` (the default) reproduces the exact baseline PersonaPlex behavior from Sections
1‑17 above, byte-for-byte.** Nothing in this section is imported or executed by the baseline server
startup; it only becomes active once you flip `ENABLE_RAG = True` and (in later notebook revisions)
run the retrieval/injection cells that build on top of it.

### Modes (see `docs/ARCHITECTURE_REPORT.md`, Section 6, for the full reasoning per mode)

| `INJECTION_MODE` | Description | Status |
|---|---|---|
| `baseline` | Mode A — no RAG, pure PersonaPlex | Always available |
| `prompt_rag` | Mode B — naive "Relevant Knowledge: ... User Question: ..." block. Intentional **negative control** — expected to underperform `persona_rag`. | Config scaffolding ready; retrieval + server wiring land in a later increment |
| `persona_rag` | Mode C — knowledge folded into the exact same `<system>...<system>` mechanism PersonaPlex uses for its own persona prompt | Config scaffolding ready; `rag/injection_manager.py` implements the underlying primitive today |
| `turn_injection` | Mode D — inject once per detected end-of-user-turn (needs `VAD_ENABLED=True`) | Implemented and concluded (Section 20 Run 4) — the burst mechanism is corruption-free, but wrapping it in `<system>` tags mid-call causes the model to re-greet instead of grounding; see `docs/MODE_C_IMPLEMENTATION_REPORT.md` Section 8 |
| `dynamic_runtime` | Mode E — inject on a recurring fixed interval throughout the call | Implemented (Section 20 Run 5) — same burst mechanism as Mode D, deliberately **without** `<system>` wrapping, to isolate whether that wrapping was the cause of Mode D’s derailment |
| `cache_aware` | Mode F — not a new mechanism: benchmarks the same connection-preserving burst as Mode C against a naive `reset_streaming()`+replay baseline, to quantify the cost of not preserving the live RingKVCache | Implemented (Section 20 Run 6) |

Only `rag/config.py`, `rag/turn_detector.py`, and `rag/injection_manager.py` exist so far (all
dependency-light, unit-tested in isolation — see `rag/tests/`). Retrieval (`rag/retriever.py`,
`rag/vector_store.py`, `rag/embeddings.py`) and the benchmark suite (`rag/benchmark.py`,
`rag/experiments.py`) are implemented incrementally, mode by mode, in subsequent notebook revisions.

In [ ]:
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ---- Configuration surface (edit these) ----------------------------------------------------
ENABLE_RAG = True                 # Master switch. False = identical to baseline Sections 1-17.
INJECTION_MODE = "baseline"        # one of: baseline | prompt_rag | persona_rag | turn_injection
                                    #         | dynamic_runtime | cache_aware
TOP_K = 5                          # retrieval depth (consumed once rag/retriever.py lands)
EMBEDDING_MODEL = "bge-small"      # bge-small | bge-base | bge-large | e5-small | e5-base | e5-large
                                    #   | sentence-transformers
VECTOR_DB = "faiss"                # faiss | chroma
BENCHMARK_MODE = True             # when True, later cells will also emit rag/benchmark.py reports
VAD_ENABLED = True                # required for INJECTION_MODE == "turn_injection"
DYNAMIC_INJECTION_INTERVAL_S = 30.0  # only used by INJECTION_MODE == "dynamic_runtime"
DYNAMIC_INJECTION_TOP_K = 2          # only used by INJECTION_MODE == "dynamic_runtime"
# ----------------------------------------------------------------------------------------------

from rag.config import InjectionMode, RAGConfig

rag_config = RAGConfig(
    enable_rag=ENABLE_RAG,
    injection_mode=InjectionMode(INJECTION_MODE),
    top_k=TOP_K,
    embedding_model=EMBEDDING_MODEL,
    vector_db=VECTOR_DB,
    benchmark_mode=BENCHMARK_MODE,
    vad_enabled=VAD_ENABLED,
    dynamic_injection_interval_s=DYNAMIC_INJECTION_INTERVAL_S,
    dynamic_injection_top_k=DYNAMIC_INJECTION_TOP_K,
    log_dir=os.path.join(WORKSPACE, "rag_logs"),
)

print(rag_config.describe())

### Sanity-check the RAG scaffolding on this pod

Runs the unit tests that ship with `rag/` (see `rag/tests/`). These tests use plain Python
stand-ins for `LMGen` and the tokenizer — they do **not** require the GPU, CUDA, or the PersonaPlex
weights to be loaded, so this cell is safe to run at any point, independent of whether the server
in Section 10 is up.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "rag/tests", "-t", "."],
    cwd=REPO_DIR, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("rag/ unit tests failed -- see output above.")
print("rag/ scaffolding self-test passed.")

### Status update: Mode C is now implemented and wired in

The retrieval stack (`rag/embeddings.py`, `rag/vector_store.py`, `rag/retriever.py`), structured
logging (`rag/logging_utils.py`), benchmarking (`rag/benchmark.py`), and the integration layer
(`rag/server_integration.py`) all now exist and are unit-tested (`rag/tests/`, 51 tests). **Mode C
(persona-compatible injection)** is wired additively into both `moshi/moshi/offline.py` (new
`--rag-enable/--rag-index/--rag-query/...` flags) and `moshi/moshi/server.py` (new
`--rag-enable/--rag-index/...` server flags + a per-connection `rag_query` query-string parameter)
-- in both cases, omitting `--rag-enable` reproduces the exact prior behavior.

The next few sections build a small test knowledge base, build a FAISS index for it, and run a
controlled A/B experiment (baseline vs. Mode C) using `moshi.offline` to get experimental evidence
that retrieval can actually change what the model says -- before moving on to Modes B/D/E/F.

Modes B/D/E/F, the live-websocket (not just offline.py) version of this experiment, and the full
Phase 8 quality/memory benchmark suite are still pending, per `docs/ARCHITECTURE_REPORT.md`.

## 19. Build the Mode C test knowledge base

`rag/data/aero_rentals_kb.json` is a small (10-document) knowledge base extending the
"AeroRentals Pro" drone-rental persona from `README.md`'s prompting guide with operational details
the bare persona prompt does **not** contain: cancellation policy, damage insurance, pilot-license
requirements, late-return fees, weather policy, pickup locations, and group discounts.

This is deliberately chosen so we can ask the model something the bare persona prompt cannot
answer correctly, and check whether retrieval + Mode C injection changes that.

This section requires two extra pip packages (`faiss-cpu`, `sentence-transformers`) -- both are
*only* needed when `ENABLE_RAG=True`; they are not part of baseline PersonaPlex's dependencies.

In [ ]:
if ENABLE_RAG:
    %pip install -q faiss-cpu sentence-transformers
    print("RAG retrieval dependencies installed.")
else:
    print("ENABLE_RAG is False -- skipping faiss-cpu/sentence-transformers install.")

In [ ]:
import json

RAG_INDEX_DIR = os.path.join(WORKSPACE, "rag_indexes")
RAG_INDEX_PATH = os.path.join(RAG_INDEX_DIR, "aero_rentals")

if ENABLE_RAG:
    from rag.build_index import build_index

    report = build_index(
        kb_path=os.path.join(REPO_DIR, "rag", "data", "aero_rentals_kb.json"),
        out_path=RAG_INDEX_PATH,
        embedding_model=EMBEDDING_MODEL,
        vector_db=VECTOR_DB,
    )
    print(json.dumps(report, indent=2))
else:
    print("ENABLE_RAG is False -- skipping index build.")

### Sanity-check retrieval before touching the model at all

Confirms the index actually returns the right document for a question the persona prompt alone
cannot answer, using only `rag/retriever.py` -- no GPU, no PersonaPlex weights involved yet.

In [ ]:
if ENABLE_RAG:
    from rag.retriever import Retriever

    check_retriever = Retriever(embedding_model=EMBEDDING_MODEL, vector_db=VECTOR_DB)
    check_retriever.load_index(RAG_INDEX_PATH)

    result = check_retriever.retrieve_context(
        "Hi, I need to cancel my drone rental tomorrow morning. What is your cancellation policy?",
        top_k=TOP_K,
    )
    for ctx, score in zip(result["contexts"], result["scores"]):
        print(f"[{score:.3f}] {ctx}")

    assert any("Cancellations made" in c for c in result["contexts"]), (
        "Expected the cancellation-policy document in the top results -- check the index build above."
    )
    print("\nRetrieval sanity check passed: the cancellation-policy document is retrieved correctly.")
else:
    print("ENABLE_RAG is False -- skipping retrieval sanity check.")

## 20. A/B/C/D/E/F experiment: baseline vs. Mode B (naive prompt) vs. Mode C (persona-compatible) vs. Mode D (turn injection) vs. Mode E (dynamic/periodic injection) vs. Mode F (cache-aware benchmark)

**Methodology.** `moshi.offline` runs a single, deterministic, scripted conversation from an input
WAV -- no live microphone or websocket needed -- which makes it the simplest reliable way to get a
side-by-side, reproducible comparison across all three:

1. **Mode A (baseline)** -- the unmodified AeroRentals persona prompt from `README.md`, no RAG.
2. **Mode C (persona-compatible RAG)** -- same persona prompt and input audio, with the
   cancellation-policy fact retrieved and injected via the same `<system>...<system>` mechanism
   PersonaPlex's own persona prompt uses (see `docs/STREAMING_AND_INJECTION_DESIGN.md`).
3. **Mode B (standard prompt RAG -- negative control)** -- the *same* retrieval as Mode C, but
   formatted as a generic chatbot-style instruction block instead:
   ```
   Relevant Knowledge:
   <retrieved facts>

   User Question:
   <query>

   Use the knowledge above when answering.
   ```
   This is the "obvious" thing someone might try if they treated PersonaPlex like an ordinary
   text-prompted chat LLM. Per `docs/ARCHITECTURE_REPORT.md` Section 6, Mode B is a deliberate
   negative control -- expected to retrieve the same facts as Mode C but ground the response less
   reliably, because the template's phrasing is out-of-distribution for what PersonaPlex was
   fine-tuned to attend to.

All three runs use the same seed, voice prompt, and input WAV
(`assets/test/aero_rentals_question_cancellation_padded.wav` -- the spoken question plus 10s of
trailing silence so the agent has room to finish a full sentence; see `docs/MODE_C_IMPLEMENTATION_REPORT.md`
Section 3c for why the padding was needed) so the **only** difference between the three transcripts
is the injection strategy (none / Mode C / Mode B).

**Why an offline, scripted question rather than live retrieval on whatever the user says:**
`docs/ARCHITECTURE_REPORT.md` (Section 6) found that PersonaPlex has **no speech-to-text of the
user's own audio anywhere in the pipeline**. Both RAG modes here are therefore connection-start
mechanisms (the query is supplied once, like `moshi.offline`'s `--rag-query` flag does), not
per-utterance ones -- see the Phase 2 implementation report for this finding in full.

4. **Mode D (turn injection)** -- nothing is injected up front. Instead, a small knowledge block (`turn_injection_top_k=2` documents, deliberately short -- see `RAGConfig.turn_injection_top_k`) is prepared and held back. The padded input WAV already contains a natural speech-then-pause pattern (the spoken question, followed by 10s of silence) -- `rag/turn_detector.py`'s energy-based detector fires once it sees that pause, which fires the prepared knowledge as **one self-contained burst** (same mechanism as Mode C, just triggered later -- see `docs/MODE_D_REDESIGN.md` for why an earlier incremental/per-tick design corrupted both the transcript and the spoken audio, and why a burst doesn't). This tests whether injecting knowledge *during* an already-started response (mid-stream, without ever resetting the connection) works as well as Mode C's up-front injection.

**Mode D's real result (see `docs/MODE_C_IMPLEMENTATION_REPORT.md` Section 8): the burst itself doesn't leak, but wrapping it in `<system>...<system>` tags mid-call appears to read to the model as "a call is starting" -- it abandons its in-progress sentence and re-greets instead of grounding in the injected facts.**

5. **Mode E (dynamic/periodic injection)** -- mechanically identical to Mode D's fix (a self-contained burst, fired later rather than at connection start), but triggered by a fixed wall-clock interval (`--rag-dynamic-injection-interval-s`) instead of a detected pause, and deliberately **not** wrapped in `<system>` tags -- a plain knowledge block, no framing at all. This isolates one specific question Mode D's result raised: was the derailment caused by the `<system>` tag itself (which the model has only ever seen at connection start), or by something inherent to mid-call injection generally? If Mode E grounds cleanly where Mode D derailed, the tag is the culprit; if it also derails, the problem is broader than the wrapping format.

**Mode E's real result (see `docs/MODE_C_IMPLEMENTATION_REPORT.md` Section 10): the `<system>`-tag hypothesis was confirmed -- Mode E's transcript does not re-greet, it stays coherent. But it also doesn't ground -- it produces the exact same wrong answer as the baseline, as if the burst had no effect. The likely reason: by the time any pause/interval trigger fires, the model has already committed to its response. Injection *timing* relative to generation, not `<system>`-tag format, looks like the real blocker for mid-call grounding -- Modes D and E are both concluded on this basis.**

6. **Mode F (cache-aware benchmark)** -- not a new injection mechanism. C/D/E already established the only thing in this project that reliably grounds (the `<system>`-wrapped burst at connection start). Mode F instead asks: *how much does it actually cost to avoid resetting the connection at all?* It fires the same connection-preserving burst as Mode C (arm 1), then measures a naive baseline that calls `reset_streaming()` (wiping the live RingKVCache) and replays the entire persona/voice prompt setup from scratch before re-injecting the same knowledge (arm 2) -- simulating what an implementation without this project's live-injection mechanism would have to do. Arm 2's latency includes the full persona+voice prompt replay cost on top of the same injection cost arm 1 paid alone, so the ratio between the two quantifies the actual benefit of this project's core premise (preserve the live cache, never reset).

In [ ]:
AERO_PERSONA_PROMPT = (
    "You work for AeroRentals Pro which is a drone rental company and your name is Tomaz Novak. "
    "Information: AeroRentals Pro has the following availability: PhoenixDrone X ($65/4 hours, "
    "$110/8 hours), and the premium SpectraDrone 9 ($95/4 hours, $160/8 hours). Deposit required: "
    "$150 for standard models, $300 for premium."
)
AERO_QUESTION_WAV = os.path.join(REPO_DIR, "assets", "test", "aero_rentals_question_cancellation_padded.wav")  # padded with 10s trailing silence so the agent has room to finish a full sentence
AERO_QUESTION_TEXT = "Hi, I need to cancel my drone rental tomorrow morning. What is your cancellation policy?"
MODE_C_SEED = 42424242
MODE_C_VOICE_PROMPT = "NATM1.pt"  # a male "natural" voice, consistent with Tomaz Novak

print("Persona prompt:", AERO_PERSONA_PROMPT)
print("Question audio:", AERO_QUESTION_WAV)
print("Question text :", AERO_QUESTION_TEXT)

### Free GPU memory before running the offline A/B experiment

**`moshi.offline` loads its own full copy of the 7B model in a separate OS process.** If the live
server from Section 10 is still running in the background, its process is *also* holding a full
model copy on the same GPU -- on a 32GB RTX 5090, two full copies (server + offline subprocess)
will not both fit, and the cell below will fail with `torch.OutOfMemoryError: CUDA out of memory`.

This cell stops the Section 10 server (if it's running) before the offline experiment, freeing its
VRAM. This is safe to run regardless of whether Section 10 was ever started. If you want the live
web UI again afterward, re-run Section 10's launch + readiness cells.

In [ ]:
import subprocess as _subprocess

if "server_proc" in globals() and server_proc.poll() is None:
    print(f"Stopping the live server (PID {server_proc.pid}) to free GPU memory for the offline Mode C experiment...")
    server_proc.terminate()
    try:
        server_proc.wait(timeout=30)
    except _subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=30)
    print("Server stopped.")
else:
    print("No live server process detected (either never started, or already stopped) -- nothing to do.")

# Quick sanity check: print current GPU memory usage so an OOM here is diagnosable immediately
# rather than surfacing as a confusing failure several cells later.
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

### Run 1 -- Mode A (baseline, no RAG)

In [ ]:
import subprocess

baseline_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", MODE_C_VOICE_PROMPT,
    "--text-prompt", AERO_PERSONA_PROMPT,
    "--input-wav", AERO_QUESTION_WAV,
    "--seed", str(MODE_C_SEED),
    "--output-wav", "mode_c_baseline.wav",
    "--output-text", "mode_c_baseline.json",
]

result_baseline = subprocess.run(baseline_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True)
print(result_baseline.stdout[-3000:])
if result_baseline.returncode != 0:
    print(result_baseline.stderr[-3000:])
    raise RuntimeError("Baseline run failed -- see output above.")

with open(os.path.join(REPO_DIR, "mode_c_baseline.json")) as f:
    baseline_tokens = json.load(f)
baseline_transcript = "".join(baseline_tokens)
print("\nBASELINE TRANSCRIPT:\n", baseline_transcript)

### Run 2 -- Mode C (persona-compatible RAG)

In [ ]:
if not ENABLE_RAG:
    raise RuntimeError("Set ENABLE_RAG = True in Section 18 and re-run the cells above before running this one.")

mode_c_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", MODE_C_VOICE_PROMPT,
    "--text-prompt", AERO_PERSONA_PROMPT,
    "--input-wav", AERO_QUESTION_WAV,
    "--seed", str(MODE_C_SEED),
    "--output-wav", "mode_c_rag.wav",
    "--output-text", "mode_c_rag.json",
    "--rag-enable",
    "--rag-index", RAG_INDEX_PATH,
    "--rag-query", AERO_QUESTION_TEXT,
    "--rag-top-k", str(TOP_K),
    "--rag-embedding-model", EMBEDDING_MODEL,
    "--rag-log-dir", rag_config.log_dir,
]

result_rag = subprocess.run(mode_c_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True)
print(result_rag.stdout[-4000:])
if result_rag.returncode != 0:
    print(result_rag.stderr[-3000:])
    raise RuntimeError("Mode C run failed -- see output above.")

with open(os.path.join(REPO_DIR, "mode_c_rag.json")) as f:
    rag_tokens = json.load(f)
rag_transcript = "".join(rag_tokens)
print("\nMODE C TRANSCRIPT:\n", rag_transcript)

### Run 3 -- Mode B (standard prompt RAG, negative control)

In [ ]:
if not ENABLE_RAG:
    raise RuntimeError("Set ENABLE_RAG = True in Section 18 and re-run the cells above before running this one.")

mode_b_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", MODE_C_VOICE_PROMPT,
    "--text-prompt", AERO_PERSONA_PROMPT,
    "--input-wav", AERO_QUESTION_WAV,
    "--seed", str(MODE_C_SEED),
    "--output-wav", "mode_b_rag.wav",
    "--output-text", "mode_b_rag.json",
    "--rag-enable",
    "--rag-index", RAG_INDEX_PATH,
    "--rag-query", AERO_QUESTION_TEXT,
    "--rag-top-k", str(TOP_K),
    "--rag-embedding-model", EMBEDDING_MODEL,
    "--rag-log-dir", rag_config.log_dir,
    "--rag-injection-mode", "prompt_rag",
]

result_mode_b = subprocess.run(mode_b_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True)
print(result_mode_b.stdout[-4000:])
if result_mode_b.returncode != 0:
    print(result_mode_b.stderr[-3000:])
    raise RuntimeError("Mode B run failed -- see output above.")

with open(os.path.join(REPO_DIR, "mode_b_rag.json")) as f:
    mode_b_tokens = json.load(f)
mode_b_transcript = "".join(mode_b_tokens)
print("\nMODE B TRANSCRIPT:\n", mode_b_transcript)

### Run 4 -- Mode D (turn injection, burst on detected pause)

In [ ]:
if not ENABLE_RAG:
    raise RuntimeError("Set ENABLE_RAG = True in Section 18 and re-run the cells above before running this one.")

mode_d_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", MODE_C_VOICE_PROMPT,
    "--text-prompt", AERO_PERSONA_PROMPT,
    "--input-wav", AERO_QUESTION_WAV,
    "--seed", str(MODE_C_SEED),
    "--output-wav", "mode_d_rag.wav",
    "--output-text", "mode_d_rag.json",
    "--rag-enable",
    "--rag-index", RAG_INDEX_PATH,
    "--rag-query", AERO_QUESTION_TEXT,
    "--rag-embedding-model", EMBEDDING_MODEL,
    "--rag-log-dir", rag_config.log_dir,
    "--rag-injection-mode", "turn_injection",
    "--rag-vad-enable",
    "--rag-turn-injection-top-k", "2",
]

result_mode_d = subprocess.run(mode_d_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True)
print(result_mode_d.stdout[-4000:])
if result_mode_d.returncode != 0:
    print(result_mode_d.stderr[-3000:])
    raise RuntimeError("Mode D run failed -- see output above.")

with open(os.path.join(REPO_DIR, "mode_d_rag.json")) as f:
    mode_d_tokens = json.load(f)
mode_d_transcript = "".join(mode_d_tokens)
print("\nMODE D TRANSCRIPT:\n", mode_d_transcript)

### Run 5 -- Mode E (dynamic/periodic injection, no <system> wrapping)

In [ ]:
if not ENABLE_RAG:
    raise RuntimeError("Set ENABLE_RAG = True in Section 18 and re-run the cells above before running this one.")

# The padded input WAV is only ~17.4s total -- Section 18's production-realistic
# DYNAMIC_INJECTION_INTERVAL_S default (30s) would never fire within it. Use a short interval here
# so the demo actually exercises at least one (likely two) fixed-interval re-injections.
MODE_E_DEMO_INTERVAL_S = 5.0

mode_e_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", MODE_C_VOICE_PROMPT,
    "--text-prompt", AERO_PERSONA_PROMPT,
    "--input-wav", AERO_QUESTION_WAV,
    "--seed", str(MODE_C_SEED),
    "--output-wav", "mode_e_rag.wav",
    "--output-text", "mode_e_rag.json",
    "--rag-enable",
    "--rag-index", RAG_INDEX_PATH,
    "--rag-query", AERO_QUESTION_TEXT,
    "--rag-embedding-model", EMBEDDING_MODEL,
    "--rag-log-dir", rag_config.log_dir,
    "--rag-injection-mode", "dynamic_runtime",
    "--rag-dynamic-injection-interval-s", str(MODE_E_DEMO_INTERVAL_S),
    "--rag-dynamic-injection-top-k", str(DYNAMIC_INJECTION_TOP_K),
]

result_mode_e = subprocess.run(mode_e_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True)
print(result_mode_e.stdout[-4000:])
if result_mode_e.returncode != 0:
    print(result_mode_e.stderr[-3000:])
    raise RuntimeError("Mode E run failed -- see output above.")

with open(os.path.join(REPO_DIR, "mode_e_rag.json")) as f:
    mode_e_tokens = json.load(f)
mode_e_transcript = "".join(mode_e_tokens)
print("\nMODE E TRANSCRIPT:\n", mode_e_transcript)


### Run 6 -- Mode F (cache-aware benchmark: burst vs. reset_and_replay)

In [ ]:
if not ENABLE_RAG:
    raise RuntimeError("Set ENABLE_RAG = True in Section 18 and re-run the cells above before running this one.")

mode_f_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", MODE_C_VOICE_PROMPT,
    "--text-prompt", AERO_PERSONA_PROMPT,
    "--input-wav", AERO_QUESTION_WAV,
    "--seed", str(MODE_C_SEED),
    "--output-wav", "mode_f_rag.wav",
    "--output-text", "mode_f_rag.json",
    "--rag-enable",
    "--rag-index", RAG_INDEX_PATH,
    "--rag-query", AERO_QUESTION_TEXT,
    "--rag-top-k", str(TOP_K),
    "--rag-embedding-model", EMBEDDING_MODEL,
    "--rag-log-dir", rag_config.log_dir,
    "--rag-injection-mode", "cache_aware",
]

result_mode_f = subprocess.run(mode_f_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True)
print(result_mode_f.stdout[-4000:])
if result_mode_f.returncode != 0:
    print(result_mode_f.stderr[-3000:])
    raise RuntimeError("Mode F run failed -- see output above.")

with open(os.path.join(REPO_DIR, "mode_f_rag.json")) as f:
    mode_f_tokens = json.load(f)
mode_f_transcript = "".join(mode_f_tokens)
print("\nMODE F TRANSCRIPT:\n", mode_f_transcript)


### Compare all six transcripts and listen to each

For Mode D, check the `[rag] strategy=...`/`turn boundary detected -> fired burst` log lines
printed by the cell above: you should see the `turn_injection (prepared; fired as a burst...)`
setup line right after the persona/voice prompt (no tokens forced yet), then later in the same
run's stdout, a `turn boundary detected -> fired burst` line once the trailing silence in the
input WAV triggers the turn-boundary detector. The transcript should **not** contain any raw
`<system>` tags or verbatim KB text spliced mid-sentence -- if it does, the burst leaked, see
`docs/MODE_D_REDESIGN.md`. The real run produced a *different* failure: no leak, but the agent
abandoned its in-progress sentence and re-greeted right at the burst point -- see
`docs/MODE_C_IMPLEMENTATION_REPORT.md` Section 8.

For Mode E, check the `dynamic-injection interval elapsed -> fired burst` log line(s) -- with
`MODE_E_DEMO_INTERVAL_S=5.0` against this ~17.4s clip, expect to see it fire once or twice. The
real run confirmed Mode E's transcript stays coherent (no re-greet, unlike Mode D) but also
doesn't ground -- it matches the baseline's wrong answer, as if the burst had no effect. See
`docs/MODE_C_IMPLEMENTATION_REPORT.md` Section 10 for why: injection *timing* relative to
generation, not `<system>`-tag format, looks like the real blocker for mid-call grounding.

For Mode F, check the `[rag] cache_aware arm 1 (burst, no reset)` / `arm 2 (reset_and_replay
baseline)` log lines -- arm 2's `injection_latency_s` should be substantially larger than arm 1's
(it pays the full persona/voice prompt replay cost on top of the same injection cost). Mode F's
transcript should ground similarly to Mode C's -- arm 2 ends by re-injecting the same knowledge,
so whatever state the main loop continues from should reflect it -- but the point of this run is
the latency comparison in Section 21, not the transcript.

In [ ]:
from IPython.display import Audio, display

print("=" * 70)
print("MODE A (baseline, no RAG):")
print(baseline_transcript)
print("=" * 70)
print("MODE C (persona-compatible RAG):")
print(rag_transcript)
print("=" * 70)
print("MODE B (standard prompt RAG, negative control):")
print(mode_b_transcript)
print("=" * 70)
print("MODE D (turn injection, burst on detected pause):")
print(mode_d_transcript)
print("=" * 70)
print("MODE E (dynamic/periodic injection, no <system> wrapping):")
print(mode_e_transcript)
print("=" * 70)
print("MODE F (cache-aware benchmark: burst vs. reset_and_replay):")
print(mode_f_transcript)
print("=" * 70)

print("\nMode A audio:")
display(Audio(os.path.join(REPO_DIR, "mode_c_baseline.wav")))
print("Mode C audio:")
display(Audio(os.path.join(REPO_DIR, "mode_c_rag.wav")))
print("Mode B audio:")
display(Audio(os.path.join(REPO_DIR, "mode_b_rag.wav")))
print("Mode D audio:")
display(Audio(os.path.join(REPO_DIR, "mode_d_rag.wav")))
print("Mode E audio:")
display(Audio(os.path.join(REPO_DIR, "mode_e_rag.wav")))
print("Mode F audio:")
display(Audio(os.path.join(REPO_DIR, "mode_f_rag.wav")))


## 21. Benchmark report: Mode C vs. Mode B vs. Mode D vs. Mode E vs. Mode F

Loads the structured log written by `rag.logging_utils.RequestLogger` during the runs above
(`<rag_config.log_dir>/requests.jsonl`) and summarizes retrieval/injection/generation/total latency
**per mode**, per `rag.benchmark`. Modes D and E each show up as *two* kinds of rows: a setup row
(retrieval only, logged via `finalize_and_log`) and one or more burst rows (one per detected pause
for D, one per elapsed interval for E, logged by `fire_*_burst`/`_async`) -- the cell below reports
both, for both modes. Mode F also logs two rows, but they're a *paired comparison*, not a
setup/completion pair: arm 1 (`cache_aware (burst, no reset...)`) and arm 2 (`cache_aware (naive
reset_and_replay baseline...)`) retrieve and inject the same knowledge -- the only difference is
arm 2 also pays for a full persona/voice prompt replay. The cell below reports both individually
and computes the ratio between them.

In [ ]:
from rag.benchmark import from_log_record, summarize
from rag.logging_utils import RequestLogger

rag_logger = RequestLogger(rag_config.log_dir)
all_records = rag_logger.read_all()

for mode_name in ("persona_rag", "prompt_rag"):
    records = [
        r for r in all_records
        if r.get("mode") == mode_name and not r.get("injection_strategy", "").startswith("skipped")
    ]
    print(f"=== mode: {mode_name} -- {len(records)} completed injection record(s) ===")
    for r in records[-3:]:
        print(json.dumps(r, indent=2))
        print("-" * 60)

    benchmarks = [from_log_record(r) for r in records]
    print(f"Summary for {mode_name}:")
    print(json.dumps(summarize(benchmarks), indent=2))
    print()

# Mode D logs two record shapes: the connection-start "setup" row (retrieval only) and one or more
# "burst" rows (one per detected turn boundary that fired -- see fire_turn_injection_burst/_async).
turn_injection_setup = [
    r for r in all_records
    if r.get("mode") == "turn_injection" and r.get("injection_strategy", "").startswith("turn_injection (prepared")
]
turn_injection_bursts = [
    r for r in all_records
    if r.get("mode") == "turn_injection" and r.get("injection_strategy", "").startswith("turn_injection (burst")
]
print(f"=== mode: turn_injection -- {len(turn_injection_setup)} setup row(s), "
      f"{len(turn_injection_bursts)} fired burst(s) ===")
for r in turn_injection_setup[-3:] + turn_injection_bursts[-3:]:
    print(json.dumps(r, indent=2))
    print("-" * 60)

burst_benchmarks = [from_log_record(r) for r in turn_injection_bursts]
print("Summary for turn_injection (fired bursts):")
print(json.dumps(summarize(burst_benchmarks), indent=2))

# Mode E logs the same two record shapes as Mode D: a connection-start "setup" row (retrieval
# only) and one or more "burst" rows (one per elapsed dynamic_injection_interval_s, fired by
# fire_dynamic_injection_burst/_async).
dynamic_runtime_setup = [
    r for r in all_records
    if r.get("mode") == "dynamic_runtime" and r.get("injection_strategy", "").startswith("dynamic_runtime (prepared")
]
dynamic_runtime_bursts = [
    r for r in all_records
    if r.get("mode") == "dynamic_runtime" and r.get("injection_strategy", "").startswith("dynamic_runtime (burst")
]
print()
print(f"=== mode: dynamic_runtime -- {len(dynamic_runtime_setup)} setup row(s), "
      f"{len(dynamic_runtime_bursts)} fired burst(s) ===")
for r in dynamic_runtime_setup[-3:] + dynamic_runtime_bursts[-3:]:
    print(json.dumps(r, indent=2))
    print("-" * 60)

dynamic_runtime_burst_benchmarks = [from_log_record(r) for r in dynamic_runtime_bursts]
print("Summary for dynamic_runtime (fired bursts):")
print(json.dumps(summarize(dynamic_runtime_burst_benchmarks), indent=2))

# Mode F logs a PAIRED comparison, not a setup/completion pair: arm 1 (cache-aware burst, no
# reset) and arm 2 (naive reset_and_replay baseline) retrieve and inject the same knowledge --
# the only difference is arm 2 also pays for a full persona/voice prompt replay.
cache_aware_arm1 = [
    r for r in all_records
    if r.get("mode") == "cache_aware" and r.get("injection_strategy", "").startswith("cache_aware (burst")
]
cache_aware_arm2 = [
    r for r in all_records
    if r.get("mode") == "cache_aware" and r.get("injection_strategy", "").startswith("cache_aware (naive")
]
print()
print(f"=== mode: cache_aware -- {len(cache_aware_arm1)} burst-arm row(s), "
      f"{len(cache_aware_arm2)} reset_and_replay-arm row(s) ===")
for r in cache_aware_arm1[-3:] + cache_aware_arm2[-3:]:
    print(json.dumps(r, indent=2))
    print("-" * 60)

if cache_aware_arm1 and cache_aware_arm2:
    arm1_latency_s = cache_aware_arm1[-1]["injection_latency_s"]
    arm2_latency_s = cache_aware_arm2[-1]["injection_latency_s"]
    print(f"Mode F comparison: cache-aware burst (arm 1) = {arm1_latency_s:.3f}s, "
          f"reset_and_replay baseline (arm 2) = {arm2_latency_s:.3f}s "
          f"-> reset_and_replay costs {arm2_latency_s / arm1_latency_s:.2f}x as much as preserving "
          f"the live RingKVCache.")
else:
    print("Mode F comparison: missing one or both arms -- check the Run 6 cell's output above.")

## 22. Production RAG Streaming Mode

Everything in Sections 18-21 was a controlled research comparison across Modes A-F using
`moshi.offline` (a single scripted run, no live websocket). This section assembles the one
mechanism that comparison actually validated -- **Mode C: connection-start, `<system>`-wrapped,
cache-preserving injection** (identical to Mode F's arm 1) -- into a standing "production" RAG
setup, demonstrated against the **live `moshi.server` websocket**, the same protocol the real web
UI speaks.

**Design decisions (deliberate, per the A-F findings above):**
- **Mode C style only.** Knowledge is retrieved once and injected as a single `<system>`-wrapped
  burst *before* the model starts generating its response to a turn -- the only approach in this
  project that reliably grounds (Section 6).
- **No reset_streaming() -- ever.** Same mechanism as Mode F's arm 1 (`RAGSession.
  fire_cache_aware_burst` / `inject_persona_compatible_knowledge`): the live RingKVCache (persona +
  voice prompt) is preserved, never wiped and replayed (Mode F's arm 2 is a deliberate non-goal
  here -- see Section 11 for why that's slower).
- **No mid-stream injection.** Modes D and E are explicitly excluded -- Section 10 found that
  forcing tokens mid-response doesn't reliably influence what the model says, regardless of
  `<system>` wrapping. Knowledge is injected exactly once, at connection start, per the existing
  `rag_query` connection parameter `moshi.server` already supports.
- **No ASR.** Per the Phase 2 architectural finding (`docs/MODE_C_IMPLEMENTATION_REPORT.md`
  Section 2), PersonaPlex has no speech-to-text of the user's own audio anywhere in the pipeline --
  there is no live transcript to retrieve against mid-call. "Once per user turn" in this
  architecture means "once per connection," exactly like Mode C/F already work; a genuinely
  per-utterance dynamic query would require bolting on a separate ASR component, out of scope here
  and explicitly excluded by the brief.

**Knowledge source.** `rag/data/text.txt` -- a single plain-text file, **not** a hand-authored
structured KB JSON. `rag.build_index.build_index_from_text_file` (new in this section) chunks it
automatically (paragraph-aware, with overlap for long paragraphs -- see `chunk_text`'s docstring)
and builds the same FAISS index format Modes B/C/D/E/F already use. To point this at your own
knowledge base, just replace the contents of `rag/data/text.txt` (or change
`PRODUCTION_TEXT_KB_PATH` below) and re-run this section -- no code changes needed.

**Why this section reuses the AeroRentals persona/voice/question from Section 20**: it's already a
proven-good question/answer pair against this exact knowledge (Section 3d's real-pod result), which
isolates the only genuinely new thing being demonstrated here -- ingesting from a *plain text file*
and running over the *live websocket* -- rather than also introducing an unvalidated new question
at the same time.

**Real-pod finding: this mechanism never engaged for actual browser conversations.** Talking to the server through the live web UI (real microphone, real voice) produced generic, ungrounded answers even with a correctly-built `text.txt` index. Root cause: `moshi.server`'s `rag_query` connection parameter only exists because *this project* added it -- the browser web UI predates it and has no field to type a query into, so it is **always empty** for every real conversation. The old code path required a truthy `rag_query` before attempting injection at all (`if self.rag_session is not None and rag_query:`), so RAG silently never fired for any real user, only for this notebook's scripted demo (which explicitly sets `rag_query=AERO_QUESTION_TEXT`).

**Fix**: `RAGSession._retrieve_for_injection` now falls back to `Retriever.retrieve_all(limit=top_k)` -- injecting up to `top_k` knowledge-base chunks regardless of relevance -- whenever the query is empty, instead of skipping injection. `moshi.server` no longer requires a truthy `rag_query` to attempt injection, and gained an optional `--rag-default-query` (operator-configured fallback query, used for real similarity search when set; the whole-KB fallback above only applies when *that* is also empty). See the new verification cell after the WebSocket demo below, which connects with `rag_query=""` -- exactly what the browser UI sends -- to prove the fallback actually grounds a connection that supplies no query at all.

**Limitation that remains, by design**: this fallback makes the model *always have* the knowledge base in context, but it cannot do true per-question retrieval live (surfacing the single most relevant chunk for whatever was actually asked) without ASR -- and Sections 8/10's findings suggest that even with ASR, a turn-boundary-triggered mid-call injection would likely arrive too late to influence the response (the model appears to commit to a response trajectory before any practical VAD/ASR pipeline could fire). For knowledge bases small enough to fit entirely in `top_k` chunks (like the 10-paragraph `text.txt` sample), this is not a real limitation in practice -- everything is in context regardless of phrasing. For much larger knowledge bases, the right fix is a smarter `--rag-default-query` (or genuinely per-question retrieval via ASR, out of scope here), not raising `--rag-top-k` to the entire KB size.

### Build the production FAISS index from `text.txt`

In [ ]:
if not ENABLE_RAG:
    raise RuntimeError("Set ENABLE_RAG = True in Section 18 and re-run the cells above before running this one.")

from rag.build_index import build_index_from_text_file

PRODUCTION_TEXT_KB_PATH = os.path.join(REPO_DIR, "rag", "data", "text.txt")  # swap in your own file here
PRODUCTION_INDEX_PATH = os.path.join(WORKSPACE, "rag_indexes", "production")

with open(PRODUCTION_TEXT_KB_PATH, encoding="utf-8") as f:
    _preview = f.read()
print(f"Knowledge source: {PRODUCTION_TEXT_KB_PATH} ({len(_preview)} chars)")
print("--- first 400 chars ---")
print(_preview[:400])
print("...")

production_index_report = build_index_from_text_file(
    text_path=PRODUCTION_TEXT_KB_PATH,
    out_path=PRODUCTION_INDEX_PATH,
    embedding_model=EMBEDDING_MODEL,
    vector_db=VECTOR_DB,
)
print()
print(json.dumps(production_index_report, indent=2))


### Start the live server in production RAG mode

Stops whatever server is currently running (Section 10's baseline, or a leftover from a previous
run of this section) and launches a fresh `moshi.server` with `--rag-enable
--rag-injection-mode persona_rag` pointed at the index just built -- the live equivalent of Mode
C/F-arm-1's connection-start injection, with the `rag_query` supplied per-connection (below) rather
than baked into the server's own startup flags.

No `--rag-default-query` is passed below -- with it omitted, any connection that supplies no `rag_query` of its own (every real browser connection) falls all the way back to injecting the whole knowledge base (`RAGSession._retrieve_for_injection`'s fallback). Pass `--rag-default-query "..."` here instead if you'd rather get real similarity-search retrieval by default for connections that don't specify their own query -- useful once your knowledge base is too large to fit entirely within `--rag-top-k` chunks.

In [ ]:
import subprocess as _subprocess

if "server_proc" in globals() and server_proc.poll() is None:
    print(f"Stopping the currently running server (PID {server_proc.pid})...")
    server_proc.terminate()
    try:
        server_proc.wait(timeout=30)
    except _subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=30)
    print("Server stopped.")
else:
    print("No live server process detected -- nothing to stop.")

PRODUCTION_LOG_PATH = os.path.join(WORKSPACE, "personaplex_server_production.log")
env = os.environ.copy()
# moshi.server is launched with cwd=REPO_DIR/moshi (below), matching Section 10's baseline
# launch -- but rag/ lives at REPO_DIR (a sibling of moshi/), so it is NOT importable from
# that cwd unless REPO_DIR is explicitly put on PYTHONPATH. The moshi.offline invocations in
# Section 20 never hit this because they use cwd=REPO_DIR directly; moshi.server's baseline
# launch in Section 10 never hit it either because it never passes --rag-enable. This is the
# fix for the "ModuleNotFoundError: No module named 'rag'" crash that --rag-enable first
# exposes here.
env["PYTHONPATH"] = REPO_DIR + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

cmd = [
    sys.executable, "-m", "moshi.server",
    "--host", SERVER_HOST, "--port", str(SERVER_PORT),
    "--rag-enable",
    "--rag-index", PRODUCTION_INDEX_PATH,
    "--rag-top-k", str(TOP_K),
    "--rag-embedding-model", EMBEDDING_MODEL,
    "--rag-log-dir", rag_config.log_dir,
    "--rag-injection-mode", "persona_rag",
]
if USE_APP_TLS:
    cmd += ["--ssl", SSL_DIR]
if USE_CPU_OFFLOAD:
    cmd += ["--cpu-offload"]

print("Launching:", " ".join(cmd))
production_log_file = open(PRODUCTION_LOG_PATH, "w")
server_proc = subprocess.Popen(
    cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, stdout=production_log_file, stderr=subprocess.STDOUT,
)
print(f"Production RAG server launched with PID {server_proc.pid}. Logs: {PRODUCTION_LOG_PATH}")


In [ ]:
import time

def _tail(path, n=60):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI directly at"
TIMEOUT_S = 900
POLL_S = 5

start = time.time()
ready = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print(_tail(PRODUCTION_LOG_PATH))
        raise RuntimeError(
            f"Production RAG server exited early with return code {server_proc.returncode}. See log above."
        )
    if READY_MARKER in open(PRODUCTION_LOG_PATH).read():
        ready = True
        break
    time.sleep(POLL_S)

print(_tail(PRODUCTION_LOG_PATH))
if not ready:
    raise TimeoutError(f"Production RAG server did not report readiness within {TIMEOUT_S}s.")
print("\nProduction RAG server is up and warmed up.")


### Run a demo query over the live WebSocket connection

`rag.ws_demo_client.run_streaming_query` connects to `/api/chat` exactly like the browser web UI
does (same query params, same Opus-encoded binary protocol -- see the module docstring), streams
the same proven AeroRentals cancellation question, and supplies `rag_query` as a per-connection
parameter -- the live-server equivalent of `moshi.offline`'s `--rag-query` flag. Audio is paced in
real time (`realtime_pacing=True`) rather than uploaded as fast as possible, so this genuinely
exercises streaming, not just correctness.

In [ ]:
from rag.ws_demo_client import run_streaming_query

scheme_ws = "wss" if USE_APP_TLS else "ws"
PRODUCTION_SERVER_URL = f"{scheme_ws}://localhost:{SERVER_PORT}/api/chat"

production_result = await run_streaming_query(
    server_url=PRODUCTION_SERVER_URL,
    voice_prompt=MODE_C_VOICE_PROMPT,
    text_prompt=AERO_PERSONA_PROMPT,
    input_wav_path=AERO_QUESTION_WAV,   # already padded with 10s trailing silence (Section 20)
    output_wav_path=os.path.join(REPO_DIR, "production_rag_response.wav"),
    output_text_path=os.path.join(REPO_DIR, "production_rag_response.json"),
    rag_query=AERO_QUESTION_TEXT,
    seed=MODE_C_SEED,
    realtime_pacing=True,
    trailing_silence_s=0.0,   # the input WAV already has its own trailing silence
    response_buffer_s=15.0,
)

print(json.dumps({k: v for k, v in production_result.items() if k != "transcript"}, indent=2))
print()
print("TRANSCRIPT:")
print(production_result["transcript"])


### Proof: retrieved chunks, grounding, and uninterrupted streaming

Reads back the same structured JSONL log every other mode in this notebook writes to
(`rag.logging_utils.RequestLogger`) and pulls the most recent `persona_rag` row -- the one this
connection's `rag_query` produced -- to show exactly what was retrieved from `text.txt` and that
injection happened once, before generation, with no `reset_streaming()` call.

In [ ]:
from rag.logging_utils import RequestLogger

production_logger = RequestLogger(rag_config.log_dir)
production_records = [
    r for r in production_logger.read_all()
    if r.get("mode") == "persona_rag" and not r.get("injection_strategy", "").startswith("skipped")
]
assert production_records, "No persona_rag log rows found -- did the demo query cell above run successfully?"
latest = production_records[-1]

print("=== Retrieved chunks from text.txt (proof of retrieval) ===")
for ctx, score in zip(latest["retrieved_contexts"], latest["retrieved_scores"]):
    print(f"[{score:.3f}] {ctx}")

print()
print("=== Injection mechanism (proof of Mode C / cache-aware, single injection per connection) ===")
print("injection_strategy:", latest["injection_strategy"])
print("injected_token_count:", latest["injected_token_count"])
print("retrieval_latency_s:", latest["retrieval_latency_s"])
print("injection_latency_s:", latest["injection_latency_s"])
print("kv_cache_status:", json.dumps(latest["kv_cache_status"], indent=2))

print()
print("=== Final answer (from the live websocket stream) ===")
print(production_result["transcript"])

print()
print("=== Streaming health ===")
print("connect_latency_s:", production_result["connect_latency_s"])
print("first_text_token_latency_s:", production_result["first_text_token_latency_s"])
print("total_duration_s:", production_result["total_duration_s"])
print("output_wav_path:", production_result["output_wav_path"])

# Best-effort grounding check -- a substring match against the retrieved facts, not a guarantee
# (PersonaPlex paraphrases rather than reciting verbatim; see Section 3d's no-show/deposit caveat).
# Inspect the printed transcript above to judge grounding for yourself; this is a quick signal,
# not the final word.
grounded_signal = any(
    keyword in production_result["transcript"].replace("PAD", "").replace("EPAD", "")
    for keyword in ("24", "50%", "refund")
)
print()
print(f"Grounding signal (best-effort substring check): {'PASS' if grounded_signal else 'NOT FOUND -- inspect transcript manually'}")

from IPython.display import Audio, display
if production_result["output_wav_path"]:
    display(Audio(production_result["output_wav_path"]))


### Verify the fix: a connection with NO query (exactly what the browser sends)

The cell above explicitly passed `rag_query=AERO_QUESTION_TEXT` -- the easy case. The actual bug
report was about the **browser web UI**, which never sends a `rag_query` at all. This cell
reproduces that exact condition (`rag_query=""`) over the same live websocket connection and
checks that injection still happens (falling back to the whole knowledge base, per the fix above)
instead of silently producing a generic, ungrounded answer.

In [ ]:
no_query_result = await run_streaming_query(
    server_url=PRODUCTION_SERVER_URL,
    voice_prompt=MODE_C_VOICE_PROMPT,
    text_prompt=AERO_PERSONA_PROMPT,
    input_wav_path=AERO_QUESTION_WAV,
    output_wav_path=os.path.join(REPO_DIR, "production_rag_no_query_response.wav"),
    rag_query="",   # <-- exactly what the browser web UI sends; this is the bug report's scenario
    seed=MODE_C_SEED,
    realtime_pacing=True,
    trailing_silence_s=0.0,
    response_buffer_s=15.0,
)

no_query_records = [
    r for r in RequestLogger(rag_config.log_dir).read_all()
    if r.get("mode") == "persona_rag" and not r.get("injection_strategy", "").startswith("skipped")
]
assert no_query_records, "No persona_rag log rows found for the no-query connection."
latest_no_query = no_query_records[-1]

print("injection_strategy:", latest_no_query["injection_strategy"])
print("user_query logged :", latest_no_query["user_query"])  # should be None -- no query was ever sent
print("retrieved_contexts:")
for ctx in latest_no_query["retrieved_contexts"]:
    print(" -", ctx)
print()
print("TRANSCRIPT (no rag_query supplied):")
print(no_query_result["transcript"])

assert latest_no_query["injection_strategy"].startswith("persona_rag"), (
    "Injection was skipped for the no-query connection -- the fallback isn't working. Check that "
    "rag/server_integration.py's _retrieve_for_injection falls back to retrieve_all()."
)
print("\nPASS: injection happened even with no rag_query -- this is what the browser UI gets by default.")


## 17. Stop the server (run only when you want to shut it down)

This is a management utility cell, **not** part of the linear startup flow — running it will terminate the
backend you just verified above. Run it deliberately when you're done with the session, not as part of a
top-to-bottom "Run All".

In [ ]:
# Intentionally NOT meant to run automatically as part of the startup sequence above.
try:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print(f"Server process {server_proc.pid} stopped.")
except NameError:
    print("No server_proc in scope -- nothing to stop.")
except subprocess.TimeoutExpired:
    server_proc.kill()
    print(f"Server process {server_proc.pid} killed after not stopping gracefully.")